# Fine-tuning FLAN-T5-Base with LoRA in Google Colab (Corrected & Stable)

This notebook trains a `FLAN-T5-base` model on a custom dataset (`combined_simplified_19830.json`) using LoRA.
It splits the dataset 70:30, disables mixed-precision `fp16` to prevent `NaN` or `0.0` loss, and generates text with penalty parameters to avoid sentence looping.

In [ ]:
# Install dependencies
!pip install -q transformers peft datasets accelerate sentencepiece torch

In [ ]:
# Clone the project repo
!git clone https://github.com/Pedri-8/Research-Intern.git
%cd Research-Intern

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq
)
from peft import (
    get_peft_model, 
    LoraConfig, 
    TaskType, 
    PeftModel
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"CUDA Available: {torch.cuda.is_available()} | Using device: {device}")

In [ ]:
# Load the combined dataset directly from Git
dataset_url = "https://raw.githubusercontent.com/Pedri-8/Research-Intern/main/output/combined_simplified_19830.json"
print(f"Loading dataset from: {dataset_url}")

raw_datasets = load_dataset("json", data_files=dataset_url)

# Split into 70% train, 30% test
split_dataset = raw_datasets["train"].train_test_split(test_size=0.3, seed=42)
print(split_dataset)

In [ ]:
# Initialize T5 Model & Tokenizer
model_id = "google/flan-t5-base" # Using base rather than small prevents context loop collapses
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

In [ ]:
# Tokenize data without static padding (delegated to DataCollatorForSeq2Seq for -100 label padding)
max_input_length = 512
max_target_length = 256

def preprocess_function(examples):
    inputs = ["simplify: " + text for text in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=examples["simplified"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = split_dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=split_dataset["train"].column_names
)
print("Preprocessing completed.")

In [ ]:
# LoRA setup
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=16,                  
    lora_alpha=32,         
    lora_dropout=0.1,      
    target_modules=["q", "v"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Train configuration & setup
training_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-base-lora-simplification",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=8,   
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,              
    fp16=False,                      # Force FP32 to prevent T5 layer norm gradients vanishing/exploding to NaN
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting training loop...")
trainer.train()

In [ ]:
# Save LoRA adapter
adapter_path = "./model_adapter_only"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"LoRA Adapter saved to {adapter_path}")

In [ ]:
# Merge adapter into T5 base to export the full merged model
print("Merging weights...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
peft_model = PeftModel.from_pretrained(base_model, adapter_path)
full_merged_model = peft_model.merge_and_unload()

full_model_path = "./full_merged_flant5_model"
full_merged_model.save_pretrained(full_model_path)
tokenizer.save_pretrained(full_model_path)
print(f"Full standalone model saved to {full_model_path}")

In [ ]:
# Copy the saved models to Google Drive for offline preservation
!cp -r ./model_adapter_only /content/drive/MyDrive/
!cp -r ./full_merged_flant5_model /content/drive/MyDrive/

In [ ]:
# Test step inference on the merged model using penalty strategies to block repetition looping
sample_text = (
    "Wellington had deploy them on the reverse slope of the ridge, "
    "where they could neither be easily seen nor easily softened up with artillery."
)

prompt = f"simplify: {sample_text}"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
merged_model_gpu = full_merged_model.to(device)

with torch.no_grad():
    # Using beams, repetition penalty and n-gram blocker to enforce diverse generation outputs
    outputs = merged_model_gpu.generate(
        **inputs, 
        max_length=128, 
        num_beams=4, 
        repetition_penalty=1.5, 
        no_repeat_ngram_size=3,
        early_stopping=True
    )

print("Original: ", sample_text)
print("Simplified:", tokenizer.decode(outputs[0], skip_special_tokens=True))